# LC11 — Forecasting with learners: features, models, quantiles (self-paced, ~50 min)

Module 2 ended with a SARIMA that beats persistence by 9%. Machine learning earns its place only if it beats *that* — on the same honest evaluation. This notebook builds the feature-based alternative: engineer features, fit learners, and — the part operators actually pay for — forecast **uncertainty** with quantile regression. Material appears in **Quiz 4**; Labs 8-9 use these working methods.

## 1. From series to table: feature engineering

A learner does not know time exists. We hand it time as *columns*: lagged load (yesterday, last week), calendar (hour as sine/cosine so 23 sits next to 0, weekday), and — SARIMA's blind spot — **temperature**. The split is temporal: October onward is the untouched future.

In [ ]:
%pip install scikit-learn pandas pyarrow matplotlib --quiet
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
df = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
y = df["ZON_MITT"].interpolate(limit=3)
X = pd.DataFrame(index=df.index)
X["lag24"], X["lag168"] = y.shift(24), y.shift(168)
X["temp"] = df["temp_mid"].interpolate(limit=3)
X["hour_sin"] = np.sin(2*np.pi*df.index.hour/24); X["hour_cos"] = np.cos(2*np.pi*df.index.hour/24)
X["dow"] = df.index.dayofweek; X["workday"] = (X["dow"] < 5).astype(int)
data = X.join(y.rename("target")).dropna()
train = data.loc[:"2025-09-30"]; test = data.loc["2025-10-01":]
Xtr, ytr = train.drop(columns="target"), train["target"]
Xte, yte = test.drop(columns="target"), test["target"]
print(f"train {len(train)} h, test {len(test)} h — split is TEMPORAL, never random")

## 2. Two learners vs the baselines

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
models = {"linear": LinearRegression(), "gbdt": HistGradientBoostingRegressor(random_state=0)}
mae = {}
for name, m in models.items():
    m.fit(Xtr, ytr)
    mae[name] = float(np.abs(m.predict(Xte) - yte).mean())
mae["persistence(24h)"] = float(np.abs(Xte["lag24"] - yte).mean())
print(pd.Series(mae).round(1).sort_values().to_string())

Two lines of sklearn beat the naive baseline clearly — mostly because of `lag24` plus temperature. Compare with your Lab 7 SARIMA number and note WHY the learner wins where it does: it is the covariates, not magic. (And a caution: this table is one 24 h-ahead-equivalent split, not the full walk-forward — Lab 9 does it properly with the Lab 7 harness.)

## 3. The forecast an operator wants: quantiles

"Tomorrow: 3 800 MW" is less useful than "with 80% probability between 3 550 and 4 100". Quantile regression fits the P10/P50/P90 *directly*, by swapping the loss function — the **pinball loss**, which penalizes being on the wrong side of the quantile asymmetrically:

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
qs = {}
for a in (0.1, 0.5, 0.9):
    g = GradientBoostingRegressor(loss="quantile", alpha=a, n_estimators=150,
                                  max_depth=3, random_state=0)
    g.fit(Xtr, ytr); qs[a] = g.predict(Xte)
import matplotlib.pyplot as plt
week = slice("2025-10-06", "2025-10-12")
idx = yte.loc[week].index
plt.figure(figsize=(10,3))
plt.fill_between(idx, pd.Series(qs[0.1], yte.index).loc[week],
                 pd.Series(qs[0.9], yte.index).loc[week], alpha=0.3, label="P10-P90")
yte.loc[week].plot(label="actual"); pd.Series(qs[0.5], yte.index).loc[week].plot(label="P50")
plt.legend(); plt.ylabel("MW"); plt.title("Probabilistic forecast, one October week");

In [ ]:
def pinball(y, q, alpha):
    d = y - q
    return float(np.mean(np.maximum(alpha*d, (alpha-1)*d)))
cover = float(((yte >= qs[0.1]) & (yte <= qs[0.9])).mean())
print(f"pinball P10 {pinball(yte, qs[0.1], .1):.1f} | P50 {pinball(yte, qs[0.5], .5):.1f} | "
      f"P90 {pinball(yte, qs[0.9], .9):.1f}")
print(f"empirical coverage of the P10-P90 band: {cover:.0%} (target: 80%)")

Two honesty checks live in that cell: pinball loss is the *proper* score for a quantile (MAE would reward the median every time), and **coverage** asks whether the band means what it claims — an 80% band that contains the truth 60% of the time is a lie with error bars.

## Self-check

In [ ]:
assert mae["gbdt"] < mae["persistence(24h)"], "the learner should beat persistence here"
assert 0.6 < cover < 0.95, "coverage catastrophically off"
print("ALL OK — learners on the table, uncertainty quantified. Lab 9: same fight, your harness.")